#### 7. Take a pandas-based script (your own, or a sample provided by your instructor) and rewrite it in PySpark, documenting at least 3 places where the pandas approach would not scale and how Spark's approach solves it.

Using Pandas Liberary

In [0]:
import pandas as pd

pandadf = pd.read_csv('/Volumes/cyntexa_dev/sales/raw/circuits.csv')
print(pandadf.to_string())

In [0]:
pandadf_filtered = pandadf[pandadf['country'] == 'USA']
pandadf_grouped = pandadf_filtered.groupby('location').agg({'location': 'count'})
print(pandadf_grouped.to_string())

Using Sparks

In [0]:
sparkdf = spark.read.csv('/Volumes/cyntexa_dev/sales/raw/circuits.csv', header=True, inferSchema=True)
sparkdf.display()

In [0]:
from pyspark.sql import functions as F

sparkdf_filtered = sparkdf.filter(F.col('country') == 'USA')
sparkdf_grouped = sparkdf_filtered.groupBy('location').agg(F.count('location').alias('count'))
sparkdf_grouped.display()

### 3 Places Where Pandas Doesn't Scale and How Spark Solves It

#### 1. **Memory Limitations - Reading Large Files**
   - **Pandas Issue**: `pd.read_csv()` loads the entire dataset into memory on a single machine. If the CSV file is larger than available RAM, the operation will fail with a MemoryError.
   - **Spark Solution**: `spark.read.csv()` reads data in a distributed manner across multiple nodes. Data is partitioned and processed in chunks, allowing Spark to handle datasets that are terabytes in size without loading everything into memory at once.

#### 2. **Single-Machine Processing - Filtering and Aggregation**
   - **Pandas Issue**: Operations like `pandadf[pandadf['country'] == 'USA']` and `groupby().agg()` execute on a single core/machine. As data size grows, processing time increases linearly and eventually becomes prohibitively slow.
   - **Spark Solution**: `sparkdf.filter()` and `groupBy().agg()` leverage distributed computing. The filtering and aggregation operations are executed in parallel across multiple worker nodes, dramatically reducing processing time for large datasets. Spark's lazy evaluation also optimizes the execution plan before running.

#### 3. **Data Shuffle and Join Operations**
   - **Pandas Issue**: When performing joins or complex aggregations, pandas must shuffle data within a single machine's memory. For large datasets with high cardinality (many unique values), this can exhaust memory and cause the process to crash or swap to disk, becoming extremely slow.
   - **Spark Solution**: Spark distributes the shuffle operation across the cluster. Data is partitioned by hash or range, and shuffle operations occur across network between nodes. While shuffles are still expensive, Spark can handle them at scale using techniques like broadcast joins for small tables and partition pruning to minimize data movement.


#### 8. Design a partitioning/write strategy (partitionBy, target file sizes) for a table that will mostly be queried by date range, and justify it using what you know about lazy evaluation and the physical plan.

In [0]:
sales_df = spark.read.csv("/Volumes/cyntexa_dev/sales/raw/sales.csv",
header = True,
inferSchema = True)

sales_df.write.mode("overwrite").partitionBy("order_date").saveAsTable("cyntexa_dev.sales.sales_partitioned")

In [0]:
from pyspark.sql.functions import col

result = spark.read.table("cyntexa_dev.sales.sales_partitioned").filter(col("order_date") >= "2023-12-03")
result.show()

This physically organizes the data on disk into separate folders/files per date value, creating a partition structure. Because the table is partitioned by order_date, when the query filters on this column, Spark uses partition pruning to skip reading files from irrelevant partitions. Instead of scanning all data, Spark only reads files from the order_date=2023-12-03 and order_date=2023-12-04 partitions.

#### 9. (Data Analyst) Using a Spark DataFrame (not SQL), reproduce a report you'd normally build in Excel/pandas — e.g., monthly revenue by category — and export the result for a dashboard.

In [0]:
from pyspark.sql import functions as F

monthly_revenue_df = sales_df.dropna().groupBy(F.col('product_id')).agg(F.sum(F.col('total_amount')).alias('revenue'))
monthly_revenue_df.write.mode('overwrite').saveAsTable('cyntexa_dev.sales.monthly_revenue_by_products')

In [0]:
%sql
select * from cyntexa_dev.sales.monthly_revenue_by_products